### 4.2.1 加权求和 Weighted Sum

假设对象函数如下，我们要完成这个函数用 Triton 实现的前向和反向传播。

In [ ]:
def weighted_sum(x, weight):
    # 这里，假设 x 具有 n 维形状 [..., D]，weight 具有 1D 形状 [D]
    return (weight * x).sum(axis=-1)

在 PyTorch 里，这行代码会启动至少两个 CUDA 内核：逐元素乘法`（weight * x）` 和按行求和`（.sum(-1)）`。
- 缺点：中间结果 (`weight * x`) 会被写回显存（HBM），然后再读出来做求和。显存读写极其耗时。

Triton 的目标：**写一个单一的 GPU 内核，把这两个操作融合在一起，数据只在高速缓存（SRAM）里倒腾，不写回显存**。

#### i 前向传播

In [ ]:
import triton
import triton.language as tl

@triton.jit
def weighted_sum_fwd(
    x_ptr, weight_ptr, # 输入指针
    output_ptr, # 输出指针
    
    x_stride_row, x_stride_dim, # 步长告诉我们如何在张量的每个轴上移动一个元素
    weight_stride_dim, # 可能是 1
    output_stride_row, # 可能是 1

    NUM_ROWS, D,
    ROWS_TILE_SIZE: tl.constexpr, D_TILE_SIZE: tl.constexpr, # 块形状必须在编译时已知
):
    # 每个实例将计算 x 的一块行的加权和。
    # `tl.program_id` 为我们提供了一种检查我们正在运行哪个线程块的方法
    row_tile_idx = tl.program_id(0)

    # 块指针为我们提供了一种从 ND 内存区域中选择并移动选择的方法。
    # 块指针必须知道：
    # - 指向张量第一个元素的指针
    # - 张量的整体形状，以处理越界访问
    # - 每个维度的步长，以正确使用内存布局
    # - 起始块的 ND 坐标，即 "offsets"
    # - 一次加载/存储的块形状
    # - 维度的顺序（从主到次轴，即 np.argsort(strides)），用于优化，在 >=Hopper 架构上支持 TMA 时需要
    x_block_ptr = tl.make_block_ptr(
        x_ptr,
        shape=(NUM_ROWS, D),
        strides=(x_stride_row, x_stride_dim),
        offsets=(row_tile_idx * ROWS_TILE_SIZE, 0),
        block_shape=(ROWS_TILE_SIZE, D_TILE_SIZE),
        order=(1, 0),       # 告诉编译器维度从主到次轴是 1 然后是 0，即行优先存储。这有助于优化内存访问（在 Hopper 架构上支持 TMA）。
    )

    weight_block_ptr = tl.make_block_ptr(
        weight_ptr,
        shape=(D,),
        strides=(weight_stride_dim,),
        offsets=(0,),
        block_shape=(D_TILE_SIZE,),
        order=(0,),
    )

    output_block_ptr = tl.make_block_ptr(
        output_ptr,
        shape=(NUM_ROWS,),
        strides=(output_stride_row,),
        offsets=(row_tile_idx * ROWS_TILE_SIZE,),
        block_shape=(ROWS_TILE_SIZE,),
        order=(0,),
    )

    # 初始化要写入的缓冲区
    output = tl.zeros((ROWS_TILE_SIZE,), dtype=tl.float32)

    for i in range(tl.cdiv(D, D_TILE_SIZE)):
        # 加载当前块指针
        # 由于 ROWS_TILE_SIZE 可能不能整除 NUM_ROWS，且 D_TILE_SIZE 可能不能整除 D，
        # 我们需要对两个维度进行边界检查
        row = tl.load(x_block_ptr, boundary_check=(0, 1), padding_option="zero") # (ROWS_TILE_SIZE, D_TILE_SIZE)
        weight = tl.load(weight_block_ptr, boundary_check=(0,), padding_option="zero") # (D_TILE_SIZE,)

        # 计算行的加权和。
        output += tl.sum(row * weight[None, :], axis=1)

        # 将指针移动到下一个块。
        # 这些是（行，列）坐标增量
        x_block_ptr = x_block_ptr.advance((0, D_TILE_SIZE)) # 沿最后一个维度移动 D_TILE_SIZE
        weight_block_ptr = weight_block_ptr.advance((D_TILE_SIZE,)) # 移动 D_TILE_SIZE

    # 将输出写入输出块指针（每行一个标量）。
    # 由于 ROWS_TILE_SIZE 可能不能整除 NUM_ROWS，我们需要边界检查
    tl.store(output_block_ptr, output, boundary_check=(0,))

#### ii 反向传播

In [ ]:
@triton.jit
def weighted_sum_backward(
    x_ptr, weight_ptr, # 输入
    grad_output_ptr, # 梯度输入
    grad_x_ptr, partial_grad_weight_ptr, # 梯度输出
    stride_xr, stride_xd,
    stride_wd,
    stride_gr,
    stride_gxr, stride_gxd,
    stride_gwb, stride_gwd,
    NUM_ROWS, D,
    ROWS_TILE_SIZE: tl.constexpr, D_TILE_SIZE: tl.constexpr,
):
    row_tile_idx = tl.program_id(0)
    n_row_tiles = tl.num_programs(0)

    # 输入
    grad_output_block_ptr = tl.make_block_ptr(
        grad_output_ptr,
        shape=(NUM_ROWS,), strides=(stride_gr,),
        offsets=(row_tile_idx * ROWS_TILE_SIZE,),
        block_shape=(ROWS_TILE_SIZE,),
        order=(0,),
    )

    x_block_ptr = tl.make_block_ptr(
        x_ptr,
        shape=(NUM_ROWS, D), strides=(stride_xr, stride_xd),
        offsets=(row_tile_idx * ROWS_TILE_SIZE, 0),
        block_shape=(ROWS_TILE_SIZE, D_TILE_SIZE),
        order=(1, 0),
    )

    weight_block_ptr = tl.make_block_ptr(
        weight_ptr,
        shape=(D,), strides=(stride_wd,),
        offsets=(0,), block_shape=(D_TILE_SIZE,),
        order=(0,),
    )

    grad_x_block_ptr = tl.make_block_ptr(
        grad_x_ptr,
        shape=(NUM_ROWS, D), strides=(stride_gxr, stride_gxd),
        offsets=(row_tile_idx * ROWS_TILE_SIZE, 0),
        block_shape=(ROWS_TILE_SIZE, D_TILE_SIZE),
        order=(1, 0),
    )

    partial_grad_weight_block_ptr = tl.make_block_ptr(
        partial_grad_weight_ptr,
        shape=(n_row_tiles, D), strides=(stride_gwb, stride_gwd),
        offsets=(row_tile_idx, 0),
        block_shape=(1, D_TILE_SIZE),
        order=(1, 0),
    )

    for i in range(tl.cdiv(D, D_TILE_SIZE)):
        grad_output = tl.load(grad_output_block_ptr, boundary_check=(0,), padding_option="zero") # (ROWS_TILE_SIZE,)

        # grad_x 的外积
        weight = tl.load(weight_block_ptr, boundary_check=(0,), padding_option="zero") # (D_TILE_SIZE,)
        grad_x_row = grad_output[:, None] * weight[None, :]
        tl.store(grad_x_block_ptr, grad_x_row, boundary_check=(0, 1))

        # 尽可能减少行数以获得 grad weight 结果
        row = tl.load(x_block_ptr, boundary_check=(0, 1), padding_option="zero") # (ROWS_TILE_SIZE, D_TILE_SIZE)
        grad_weight_row = tl.sum(row * grad_output[:, None], axis=0, keep_dims=True)
        tl.store(partial_grad_weight_block_ptr, grad_weight_row, boundary_check=(1,)) # 维度 0 永远不会越界

        # 将指针移动到 D 维度上的下一个块
        x_block_ptr = x_block_ptr.advance((0, D_TILE_SIZE))
        weight_block_ptr = weight_block_ptr.advance((D_TILE_SIZE,))
        partial_grad_weight_block_ptr = partial_grad_weight_block_ptr.advance((0, D_TILE_SIZE))
        grad_x_block_ptr = grad_x_block_ptr.advance((0, D_TILE_SIZE))

In [ ]:
import torch
from einops import rearrange

class WeightedSumFunc(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, weight):
        # 缓存 x 和 weight，以便在反向传播中使用，届时我们只接收输出张量的梯度，
        # 并且需要计算关于 x 和 weight 的梯度。
        D, output_dims = x.shape[-1], x.shape[:-1]

        # 将输入张量重塑为 2D
        input_shape = x.shape
        x = rearrange(x, "... d -> (...) d")

        ctx.save_for_backward(x, weight)

        assert len(weight.shape) == 1 and weight.shape[0] == D, "Dimension mismatch"
        assert x.is_cuda and weight.is_cuda, "Expected CUDA tensors"
        assert x.is_contiguous(), "Our pointer arithmetic will assume contiguous x"

        ctx.D_TILE_SIZE = triton.next_power_of_2(D) // 16 # 大约循环 16 次嵌入维度
        ctx.ROWS_TILE_SIZE = 16 # 每个线程一次处理 16 个批次元素
        ctx.input_shape = input_shape

        # 需要初始化空的结果张量。注意这些元素不一定是 0！
        y = torch.empty(output_dims, device=x.device)

        # 启动我们的内核，使用 1D 网格中的 n 个实例。
        n_rows = y.numel()
        weighted_sum_fwd[(triton.cdiv(n_rows, ctx.ROWS_TILE_SIZE),)](
            x, weight,
            y,
            x.stride(0), x.stride(1),
            weight.stride(0),
            y.stride(0),
            NUM_ROWS=n_rows, D=D,
            ROWS_TILE_SIZE=ctx.ROWS_TILE_SIZE, D_TILE_SIZE=ctx.D_TILE_SIZE,
        )

        return y.view(input_shape[:-1])

    @staticmethod
    def backward(ctx, grad_out):
        x, weight = ctx.saved_tensors
        ROWS_TILE_SIZE, D_TILE_SIZE = ctx.ROWS_TILE_SIZE, ctx.D_TILE_SIZE # 这些不必相同
        n_rows, D = x.shape

        # 我们的策略是让每个线程块首先写入一个部分缓冲区，
        # 然后我们对该缓冲区进行归约以获得最终梯度。
        partial_grad_weight = torch.empty((triton.cdiv(n_rows, ROWS_TILE_SIZE), D), device=x.device, dtype=x.dtype)
        grad_x = torch.empty_like(x)

        weighted_sum_backward[(triton.cdiv(n_rows, ROWS_TILE_SIZE),)](
            x, weight,
            grad_out,
            grad_x, partial_grad_weight,
            x.stride(0), x.stride(1),
            weight.stride(0),
            grad_out.stride(0),
            grad_x.stride(0), grad_x.stride(1),
            partial_grad_weight.stride(0), partial_grad_weight.stride(1),
            NUM_ROWS=n_rows, D=D,
            ROWS_TILE_SIZE=ROWS_TILE_SIZE, D_TILE_SIZE=D_TILE_SIZE,
        )
        grad_weight = partial_grad_weight.sum(axis=0)
        return grad_x, grad_weight